# 201 · Self-describing vs schema-dependent

Companion to [Self-describing vs schema](https://leo-gan.github.io/GLD.SerializerBenchmark/theory/201/self-describing-vs-schema-dependent/).

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/leo-gan/GLD.SerializerBenchmark/blob/master/docs/theory/notebooks/201/self_describing_vs_schema.ipynb)

**Goal:** measure how field names/type tags travel with JSON/MessagePack vs field numbers on a mini Protobuf-style encoding.

> **Honesty banner:** sizes and timings here are **illustrative**. Suite [Results](https://leo-gan.github.io/GLD.SerializerBenchmark/) own harness truth. Compare within one language and paradigm—not global format rankings.



In [ ]:
import json

RECORD = {"user_id": 42, "name": "Ada", "balance": 100}

try:
    import msgpack
    HAS_MSGPACK = True
except ImportError:
    HAS_MSGPACK = False
    print("pip install msgpack  # optional for MessagePack cells")



## 1. JSON — names on every message



In [ ]:
def hex_bytes(b: bytes) -> str:
    return " ".join(f"{x:02x}" for x in b)

j = json.dumps(RECORD, separators=(",", ":")).encode("utf-8")
print(j.decode())
print("nbytes", len(j))
print("hex", hex_bytes(j)[:80], "…")
# Field names are UTF-8 substrings on the wire:
for key in RECORD:
    assert key.encode() in j
print("OK: each key appears as UTF-8 in the payload")



## 2. MessagePack — type tags (+ keys if map)



In [ ]:
if not HAS_MSGPACK:
    print("SKIP msgpack")
else:
    packed = msgpack.packb(RECORD, use_bin_type=True)
    print("nbytes", len(packed))
    print("hex", hex_bytes(packed))
    # keys still present for map encoding
    for key in RECORD:
        assert key.encode() in packed
    print("OK: map keys still travel with MessagePack maps")
    # array form drops names — schema-like discipline without IDL
    as_array = msgpack.packb([RECORD["user_id"], RECORD["name"], RECORD["balance"]], use_bin_type=True)
    print("as array nbytes", len(as_array), "hex", hex_bytes(as_array))
    for key in RECORD:
        assert key.encode() not in as_array
    print("OK: array form has no field names (order is the contract)")



## 3. Schema-dependent sketch (field numbers, no names)

Same rules as 401 MiniUser subset—teaching only.



In [ ]:
def encode_varint(u: int) -> bytes:
    out = bytearray()
    while u > 0x7F:
        out.append((u & 0x7F) | 0x80)
        u >>= 7
    out.append(u & 0x7F)
    return bytes(out)


def encode_key(fn: int, wt: int) -> bytes:
    return encode_varint((fn << 3) | wt)


def encode_record_pb_style(r: dict) -> bytes:
    # 1:user_id varint, 2:name string, 3:balance as fixed64 bits of float64 (toy)
    import struct
    out = bytearray()
    out += encode_key(1, 0) + encode_varint(int(r["user_id"]))
    name = r["name"].encode()
    out += encode_key(2, 2) + encode_varint(len(name)) + name
    out += encode_key(3, 1) + struct.pack("<d", float(r["balance"]))  # wire type 1 = 8 bytes
    return bytes(out)


pb = encode_record_pb_style(RECORD)
print("pb-style nbytes", len(pb), "hex", hex_bytes(pb))
for key in RECORD:
    assert key.encode() not in pb
print("OK: field names absent; numbers 1/2/3 carry identity via shared schema")



## Size table (this payload only)



In [ ]:
rows = [("JSON", len(j)), ("pb-style sketch", len(pb))]
if HAS_MSGPACK:
    rows.insert(1, ("MessagePack map", len(msgpack.packb(RECORD, use_bin_type=True))))
    rows.insert(2, ("MessagePack array", len(msgpack.packb(list(RECORD.values()), use_bin_type=True))))
for name, n in rows:
    print(f"{name:20} {n:4} bytes")



## Takeaways

- **Self-describing:** meaning partly in the payload (names/tags).
- **Schema-dependent:** meaning in the shared contract; wire is denser and opaque alone.
- Spectrum: JSON → MessagePack map → MessagePack array → Protobuf field numbers.

**Next:** [Schema evolution](./schema_evolution.ipynb)

